# 第 2 周练习 —— 多模态技术问答助手（聊天 + 语音 + 图像 + 工具）

## 练习目标

做一个 Gradio 技术问答助手：

- **聊天**：`gpt-4.1-mini` 解释代码（Markdown）
- **工具（Tool Calling）**：`check_sintax` 用 `ast.parse` 校验 Python 语法
- **语音**：`gpt-4o-mini-tts` 把回答读出来
- **图像**：若工具拿到代码片段，用 `dall-e-3` 画一张 pop-art 风格图

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Tools / function calling | `tools=` + `finish_reason=="tool_calls"` 循环 |
| 多模态 | Chat + TTS + Images 同屏输出 |
| Gradio Blocks | `Chatbot` / `Audio` / `Image` 事件链 |

## 怎么跑

1. `.env` 配置有效的 `OPENAI_API_KEY`（`sk-proj-...`）
2. 自上而下运行；最后一格 `ui.launch(inbrowser=True)` 打开界面
3. 提问时可贴一段 Python 代码，观察工具校验、朗读与配图


In [41]:
# ========== 导入：多模态助手所需依赖 ==========

# 标准库 os：读环境变量里的 API Key
import os
# 从 dotenv 导入 load_dotenv：加载 .env，避免密钥写进笔记本
from dotenv import load_dotenv
# OpenAI 客户端：聊天、图像、语音都走这一套 SDK
from openai import OpenAI
# base64：把 DALL·E 返回的 b64_json 解码成二进制图片
import base64
# BytesIO：把字节流当成「内存文件」给 PIL 打开
from io import BytesIO
# PIL.Image：把解码后的字节变成可展示的图像对象
from PIL import Image
# json：解析 tool_call 里的 arguments 字符串
import json
# ast：用语法树解析校验 Python 代码片段是否合法
import ast
# gradio：搭 Web UI（聊天框、音频、图片）
import gradio as gr


In [42]:
# ========== 常量：模型名字集中管理 ==========

# 聊天用的小模型（便宜、适合解释类问答）
MODEL_GPT = 'gpt-4.1-mini'
# 图像生成模型（常量定义；artist() 里仍写死字符串，保持原样）
MODEL_IMAGE = 'dall-e-3'
# 文本转语音（TTS）模型
MODEL_TTS = 'gpt-4o-mini-tts'


In [43]:
# ========== 环境 + OpenAI 客户端 + System Prompt ==========

# 加载 .env（override=True 覆盖已有同名环境变量）
load_dotenv(override=True)
# 读取 OpenAI API Key
api_key = os.getenv('OPENAI_API_KEY')

# 简单校验：存在、以 sk-proj- 开头、长度足够（print 文案原文保留）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    print("OpenAPI API Key is missing.")

# 创建默认 OpenAI 客户端（内部会使用环境变量中的密钥）
openai = OpenAI()

# System prompt：告诉模型如何解释代码、何时调用工具（英文原文不可翻译）
technical_explainer_system_prompt = """
You will be asked code technical questions.
You can call a tool called check_sintax to check if python code is valid.
You will be provided with python code snippets,
explain every line and what it does.
Respond in markdown without html tags.
"""


OpenAI API Key was correctly setup.


In [44]:
# ========== 图像：根据代码片段生成 pop-art 配图 ==========

# artist：输入代码字符串 → 返回 PIL Image
def artist(code_snippet):
    # 调用 Images API；prompt / model / size 等参数保持原文
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image of code snippet {code_snippet}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    # 取出 base64 编码的图片数据
    image_base64 = image_response.data[0].b64_json
    # 解码成原始字节
    image_data = base64.b64decode(image_base64)
    # 用 BytesIO + PIL 打开为 Image 对象，供 Gradio Image 组件显示
    return Image.open(BytesIO(image_data))


In [45]:
# ========== 语音：把助手回复转成音频字节 ==========

# talker：文本 → TTS 二进制音频（供 Gradio Audio 播放）
def talker(message):
    # Speech API：指定 TTS 模型、音色 voice、输入文本
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",
      input=message
    )
    # 返回音频内容字节（response.content）
    return response.content


In [46]:
# ========== 工具实现：用 ast.parse 校验 Python 语法 ==========

# is_valid_python：给模型 tool 回传的人类可读结果字符串（英文原文保留）
def is_valid_python(code_snippet):
    try:
        # 能 parse 成功 ⇒ 语法合法（不执行代码，只做语法树）
        ast.parse(code_snippet)
        return "Success! The provided code snippet is valid python code."
    except SyntaxError:
        # 抛 SyntaxError ⇒ 语法不合法
        return "Error! The provided snippet is not valid python code!"


In [47]:
# ========== 工具 schema：告诉模型可以调用哪个函数、参数长什么样 ==========

# OpenAI function-calling 的 JSON 描述（name/description 拼写 sintax 保持原样）
valid_python_code_function = {
    "name": "check_sintax",
    "description": "Check if the provided python code snippet has correct sintax.",
    "parameters": {
        "type": "object",
        "properties": {
            "snippet": {
                "type": "string",
                "description": "The python code snippet",
            },
        },
        "required": ["snippet"],
        "additionalProperties": False
    }
}
# 包装成 tools 列表，供 chat.completions.create(..., tools=tools)
tools = [{"type": "function", "function": valid_python_code_function}]
# 单元格末尾表达式：在笔记本里预览 tools 结构
tools


[{'type': 'function',
  'function': {'name': 'check_sintax',
   'description': 'Check if the provided python code snippet has correct sintax.',
   'parameters': {'type': 'object',
    'properties': {'snippet': {'type': 'string',
      'description': 'The python code snippet'}},
    'required': ['snippet'],
    'additionalProperties': False}}}]

In [51]:
# ========== 处理 tool_calls：执行本地校验并组装 tool 消息 ==========

# 输入：助手 message（含 tool_calls）；输出：tool 回复列表 + 抽到的代码片段列表
def handle_tool_calls_and_return_snippets(message):
    # responses：要追加回 messages 的 role=tool 消息
    responses = []
    # snippets：从参数里抽出的代码，后面可能拿去画图
    snippets = []
    # 一次回复里可能有多个 tool_call，逐个处理
    for tool_call in message.tool_calls:
        # arguments 是 JSON 字符串，先 loads 成字典
        arguments = json.loads(tool_call.function.arguments)
        # 取出模型传入的 snippet 参数
        snippet = arguments.get('snippet')
        snippets.append(snippet)
        # 本地执行语法检查，得到要回传给模型的 content
        valid_python_code = is_valid_python(snippet)
        # 组装 tool 角色消息；tool_call_id 必须与请求里的 id 对应
        responses.append({
            "role": "tool",
            "content": valid_python_code,
            "tool_call_id": tool_call.id
        })
    return responses, snippets


In [52]:
# ========== 核心 chat：工具循环 → 文本回复 → TTS → 可选配图 ==========

def chat(history):
    # 把 Gradio messages 历史规整成仅含 role/content 的字典列表
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史对话
    messages = [{"role": "system", "content": technical_explainer_system_prompt}] + history
    # 第一次补全：允许 tools
    response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)
    snippets = []
    image = None

    # 若模型要求调工具，则进入循环：执行工具 → 把结果塞回 messages → 再问模型
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, snippets = handle_tool_calls_and_return_snippets(message)
        # 先把带 tool_calls 的 assistant message 放进上下文
        messages.append(message)
        # 再扩展所有 tool 回复
        messages.extend(responses)
        # 带着工具结果继续补全
        response = openai.chat.completions.create(model=MODEL_GPT, messages=messages, tools=tools)

    # 最终自然语言回复
    reply = response.choices[0].message.content
    # 追加到聊天历史，供 UI 显示
    history += [{"role":"assistant", "content":reply}]

    # 把整段回复转成语音
    voice = talker(reply)

    # 若本轮工具拿到了代码片段，用第一段去生成配图
    if snippets:
        image = artist(snippets[0])

    # 返回：更新后的历史、音频、图片（给 Gradio outputs）
    return history, voice, image


In [53]:
# ========== Gradio UI：输入框 → 追加用户消息 → 再跑 chat() ==========

# 回调：清空输入框，并把用户消息 append 到 chatbot 历史
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# Blocks 布局：左聊天、右图；下一行音频；再下一行输入框
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        # autoplay=True：拿到音频后自动播放
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # 事件链：submit 先更新历史，.then 再调用 chat 填满 chatbot/audio/image
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

# 启动 Gradio；inbrowser=True 尝试自动打开浏览器
ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
